# 1. Cells and selections

I follow one simulated long-lived particle (LLP) jet back to its calorimeter
cells. Then I use their timing to calculate Emil's score for beam-induced
background (BIB). Later I use unpaired bunch crossings as a BIB-enriched
sample and truth-matched LLP jets as signal.

Use the **PPP first look** kernel. The ROOT files go in `DataForEmre/`.

In [ ]:
from pathlib import Path

import awkward as ak
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import uproot
from IPython.display import display
from matplotlib.patches import Circle

project = Path.cwd()
if project.name == "notebooks":
    project = project.parent

llp_folder = project / "DataForEmre/collision/LLP/N1N2_rpv_1400_10ns"
llp_files = sorted(llp_folder.glob("*.root"))
assert llp_files, f"No LLP ROOT files in {llp_folder}"
llp_file = llp_files[0]

## Choose signal jets

One row in `JetContent` is an event, which can contain several jets.
I use `jet_is_llp_matched == 1` to pick signal jets. This flag comes
from the simulation. I also require valid values, $p_T\ge 30$ GeV,
$|\eta|<1.7$, and some energy in both the Tile and liquid-argon calorimeters.
Here $p_T$ is momentum across the beam and $\eta$ describes direction.
Momentum is stored in MeV and the corrected ATLAS time is in ns.

In [ ]:
fields = ["jet_is_llp_matched", "jet_pt", "jet_eta", "jet_time_ATLAS", "jet_TileEnergyFrac"]
with uproot.open(llp_file) as source:
    jets = source["JetContent"].arrays(fields, library="ak")

matched = jets["jet_is_llp_matched"] == 1
pt, eta, time, tile = (jets[name] for name in fields[1:])

def valid_number(values):
    return np.isfinite(values) & (values != -999999)

In [ ]:
keep = ak.ones_like(pt, dtype=bool)
counts = {"All jets": int(ak.sum(keep))}
cuts = {
    "LLP matched": matched,
    "Valid momentum, direction and time": valid_number(pt) & valid_number(eta) & valid_number(time),
    "pT ≥ 30 GeV": pt / 1000 >= 30,
    "|eta| < 1.7": abs(eta) < 1.7,
    "Energy in Tile and LAr": (tile > 0) & (tile < 1),
}
for name, passes in cuts.items():
    keep = keep & passes
    counts[name] = int(ak.sum(keep))
display(pd.Series(counts, name="Jets left in the first file"))

## Find one jet's cells

The file keeps the full event's cells in order. Each jet owns some
clusters and each cluster owns some cells. If the cluster counts per jet
are `[2, 1]` and the cell counts per cluster are `[3, 2, 4]`, the first
jet gets cells `[0:5]` and the next gets `[5:9]`. I trace the first
truth-matched jet as a worked example.

In [ ]:
rows = ak.where(ak.any(matched, axis=1))[0]
assert len(rows), "No matched LLP jet in the first file."
event_index = int(rows[0])
jet_index = int(ak.where(matched[event_index])[0][0])

fields = [
    "runNumber", "eventNumber", "jet_n_clusters", "cluster_n_cells",
    "jet_eta", "jet_phi", "cell_eta", "cell_phi", "jet_llp_idx", "llp_daughter_parent",
    "llp_daughter_calo_eta", "llp_daughter_calo_phi",
    "jet_llp_n_aligned_cells", "jet_llp_n_aligned_clusters",
    "cell_time", "cell_time_sigma", "cell_x", "cell_y", "cell_z", "jet_BIB_score",
]
with uproot.open(llp_file) as source:
    event = source["JetContent"].arrays(
        fields, entry_start=event_index, entry_stop=event_index + 1, library="ak",
    )[0]

print(f"Run {event['runNumber']}, event {event['eventNumber']}, jet {jet_index}")

In [ ]:
cluster_counts = event["jet_n_clusters"]
cell_counts = event["cluster_n_cells"]
cluster_start = int(ak.sum(cluster_counts[:jet_index]))
cluster_stop = cluster_start + int(cluster_counts[jet_index])
cell_start = int(ak.sum(cell_counts[:cluster_start]))
cell_stop = int(ak.sum(cell_counts[:cluster_stop]))
cell_slice = slice(cell_start, cell_stop)

assert int(ak.sum(cluster_counts)) == len(cell_counts)
assert int(ak.sum(cell_counts)) == len(event["cell_eta"])
print(f"Clusters [{cluster_start}:{cluster_stop}], cells [{cell_start}:{cell_stop}]")

## Compare with the LLP decay

I put the jet at $(\eta,\phi)=(0,0)$ in the plot by subtracting its
angles from the cells and the projected decay particles. I wrap the
difference in $\phi$ to $[-\pi,\pi)$ because it is an angle around the
beam. The circles show $\Delta R=\sqrt{\Delta\eta^2+\Delta\phi^2}=0.08$,
the distance used to count aligned cells.

In [ ]:
def wrap(angle):
    return (angle + np.pi) % (2 * np.pi) - np.pi

axis_eta = float(event["jet_eta"][jet_index])
axis_phi = float(event["jet_phi"][jet_index])
cell_eta = ak.to_numpy(event["cell_eta"][cell_slice]) - axis_eta
cell_phi = wrap(ak.to_numpy(event["cell_phi"][cell_slice]) - axis_phi)

parent = int(event["jet_llp_idx"][jet_index])
assert parent >= 0, "This jet has no assigned LLP."
daughters = event["llp_daughter_parent"] == parent
d_eta = ak.to_numpy(event["llp_daughter_calo_eta"][daughters])
d_phi = ak.to_numpy(event["llp_daughter_calo_phi"][daughters])
assert len(d_eta) and np.all(valid_number(d_eta) & valid_number(d_phi))
daughter_eta = d_eta - axis_eta
daughter_phi = wrap(d_phi - axis_phi)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4), layout="constrained")
for ax in axes:
    ax.scatter(cell_eta, cell_phi, s=14, label="Jet cells")
    ax.scatter(daughter_eta, daughter_phi, c="tab:red", marker="x", label="Projected daughters")
    ax.scatter(0, 0, c="black", marker="+", label="Jet axis")
    for a, b in zip(daughter_eta, daughter_phi):
        ax.add_patch(Circle((a, b), 0.08, fill=False, color="tab:red", linestyle="--"))
    ax.set(xlabel=r"$\Delta\eta$", ylabel=r"$\Delta\phi$ [rad]", aspect="equal")
    ax.grid(alpha=0.2)

axes[0].set_title("All projections")
axes[1].set(title="Near the jet", xlim=(-0.45, 0.45), ylim=(-0.45, 0.45))
axes[1].legend(fontsize=8)
plt.show()

A cell counts once if it is within 0.08 of any projected daughter. I
check the stored number of aligned cells and the number of clusters with
at least 30% aligned cells. In the version of the matching code I checked,
the minimum for a whole jet is 20%. I still need to confirm which version
produced these files.

[LLP matching code](https://gitlab.cern.ch/ezaya/athjetconstituentsalg-public/-/blob/a7c625c364893bbc90248fbb98a5251c1a54fdeb/AthJetConstituents/src/AthProcessing.cxx)

In [ ]:
distance_sq = (cell_eta[:, None] - daughter_eta)**2 + wrap(cell_phi[:, None] - daughter_phi)**2
aligned = (distance_sq < 0.08**2).any(axis=1)
sizes = cell_counts[cluster_start:cluster_stop]
aligned_per_cluster = ak.sum(ak.unflatten(aligned, sizes), axis=1)
aligned_clusters = int(ak.sum(aligned_per_cluster / sizes >= 0.30))

assert int(aligned.sum()) == int(event["jet_llp_n_aligned_cells"][jet_index])
assert aligned_clusters == int(event["jet_llp_n_aligned_clusters"][jet_index])
print(f"Aligned cells: {aligned.sum()} / {len(aligned)}. Aligned clusters: {aligned_clusters}.")

## Reproduce Emil's timing score

Both a slow LLP and BIB can give a late jet. I compare a flat timing
template with a particle moving along the beam. The beam template is
$g_i=(s z_i-\sqrt{x_i^2+y_i^2+z_i^2})/c$, where $s=\pm1$ is set by
how the cell time varies with $z$. The square-root term subtracts the
prompt flight from the interaction point. I use mm for position and ns
for time. Simone applied this timing comparison within clusters. Emil's
score uses cells across a jet.

[Simone Gallareto, thesis equations 7.6–7.10](https://kth.diva-portal.org/smash/get/diva2%3A1985126/FULLTEXT01.pdf)

In [ ]:
fields = ["cell_time", "cell_time_sigma", "cell_x", "cell_y", "cell_z"]
raw = np.column_stack([ak.to_numpy(event[name][cell_slice]) for name in fields])
usable = valid_number(raw).all(axis=1) & (raw[:, 1] > 0)
t, sigma, x, y, z = raw[usable].T
assert len(t) >= 4, "The timing fit needs at least four valid cells."
weights = 1 / sigma**2

def weighted_mean(values):
    return np.average(values, weights=weights)

covariance = weighted_mean((t - weighted_mean(t)) * (z - weighted_mean(z)))
direction = 1 if (covariance > 0 or (covariance == 0 and weighted_mean(z) >= 0)) else -1
beam = (direction * z - np.hypot(np.hypot(x, y), z)) / 299.792458

Each cell gets weight $w_i=1/\sigma_i^2$ from its timing uncertainty.
For each template I subtract the weighted mean of $t_i-g_i$ to allow
a common offset. The weighted mean square left over is $V_g$. The flat
template has $g_i=0$.

$$S = \frac{V_{\rm flat}}{V_{\rm flat}+V_{\rm beam}}.$$

A score above 0.5 means the beam fit has less spread. This example gives
about 0.214, so the flat fit does better.

In [ ]:
def residual_variance(template):
    residual = t - template
    return weighted_mean((residual - weighted_mean(residual))**2)

flat_var = residual_variance(0)
beam_var = residual_variance(beam)
score = flat_var / (flat_var + beam_var)
stored = float(event["jet_BIB_score"][jet_index])
np.testing.assert_allclose(score, stored, rtol=0, atol=1e-7)
print(f"Calculated score: {score:.5f}. Stored score: {stored:.5f}.")

[Train and compare](03_train_and_compare.ipynb) applies the selections across files
and checks the learned models against Emil on the same delayed jets.